<font face="B Nazanin" size=7>
    <div dir="rtl">
        <p>
        Byte Pair Encoding (BPE) یک الگوریتم توکنایزیشن است که برای فشرده‌سازی متن توسعه یافته بود و بعداً توسط OpenAI  برای آموزش مدل GPT استفاده شد.

**Character-Level BPE**

 BPE یک الگوریتم حریصانه (Greedy Algorithm) است. این به این معنی است که در هر گام، محلی‌ترین و بهینه‌ترین تصمیم را می‌گیرد (ادغام پرتکرارترین جفت) و این کار را تا رسیدن به هدف نهایی (اندازه واژگان مشخص) ادامه می‌دهد، بدون اینکه به تأثیرات بلندمدت آن فکر کند.

ورودی: یک پیکره (Corpus) از متون و یک عدد صحیح V که اندازه نهایی واژگان مطلوب است.

واژگان پایه: واژگان اولیه شامل تمام کاراکترهای منحصربه‌فرد در پیکره است. برای مثال، اگر پیکره فقط شامل "cat" و "car" باشد، واژگان پایه {c, a, t, r} خواهد بود.

تکرار (Iteration):

*   در هر گام، الگوریتم تمام جفت‌های متوالی مانند ((c, a), (a, t), (c, a), (a, r)) را در کل پیکره شمارش می‌کند.

*   جفتی که بیشترین تکرار را دارد، انتخاب می‌شود.

*   یک توکن جدید با ادغام این جفت ایجاد می‌شود (مثلاً اگر  (t, h)پرتکرارترین باشد، توکن  thایجاد می‌شود.)

*   تمام نمونه‌های آن جفت در پیکره با توکن جدید جایگزین می‌شوند.

*   این فرآیند تا زمانی که اندازه واژگان به V برسد، تکرار می‌شود

در زمان توکن‌سازی متن جدید، BPE آن را با حریصانه‌ترین (طولانی‌ترین) زیرواژه‌های ممکن از سمت چپ به راست تقسیم می‌کند. به عنوان مثال، اگر tokenization را بخواهد توکن‌سازی کند و در واژگانش  tokenو  tokوجود داشته باشد،token  را انتخاب می‌کند.

اگر کاراکتری در واژگان پایه نباشد، به یک توکن ناشناخته ( [UNK]یا"unknown token" ) تبدیل می‌شود. این مشکل در مدل‌هایی مانند GPT-2  و RoBERTa با استفاده ازBPE  در سطح بایت (byte-level BPE) حل شده است تا اطمینان حاصل شود که تمام کاراکترهای ممکن در واژگان پایه گنجانده شده‌اند.

**Byte-Level BPE**

Byte-Level BPE این مشکل را با تغییر واحد پایه از کاراکترهای یونیکد به بایت‌ها حل می‌کند. هر کاراکتر یونیکد، در نهایت به یک یا چند بایت در سیستم‌های کامپیوتری تبدیل می‌شود (با استفاده از یک کدگذاری مانند UTF-8). تعداد بایت‌های ممکن فقط ۲۵۶ تا است (از ۰ تا ۲۵۵).

تبدیل به بایت: اولین قدم در BPE در سطح بایت، تبدیل کل متن به رشته‌ای از بایت‌ها است. برای مثال، کلمه Hello در UTF-8 به بایت‌های [72, 101, 108, 108, 111] تبدیل می‌شود. حتی یک ایموجی 🌟 هم به چندین بایت تبدیل می‌شود، مثلاً [226, 156, 172].

یادگیری ادغام‌ها: الگوریتم BPE حالا نه بر روی H و e، بلکه بر روی جفت بایت‌های متوالی مانند (72, 101) عمل می‌کند. این الگوریتم، پرکاربردترین جفت‌های بایت را پیدا کرده و آن‌ها را به یک توکن جدید ادغام می‌کند. این فرآیند ادامه می‌یابد تا به اندازه واژگان مطلوب برسد.
        </p>
    </div>

In [115]:
import re
from collections import defaultdict

In [116]:
corpus = ["The old clock on the wall ticked with a steady tick-tock.",
          "The clock had been there for a long, long time.",
          "It was a clock that kept time perfectly, a clock that never failed.",
          "Its ticking was a familiar sound, a comforting sound in the quiet room.",
          "The clock watched as the world changed, but the clock itself remained the same."
          ]

<font face="B Nazanin" size=4>
    <div dir="rtl">
        <p>
         ابتدا تابع پیش‌توکنایزر به نام simple_pre_tokenize تعریف کردیم که متن را بر اساس فضاها و علائم نگارشی به کلمات اولیه تقسیم می‌کند.
         سپس، فراوانی هر کلمه محاسبه و در دیکشنری word_freqs ذخیره شد.
        </p>
    </div>

In [117]:
def simple_pre_tokenize(text):
    tokens = re.findall(r"\w+|[^\w\s]", text)
    return tokens

word_freqs = defaultdict(int)

for text in corpus:
    new_words = simple_pre_tokenize(text)
    for word in new_words:
        word_freqs[word] += 1

In [118]:
print(word_freqs)

defaultdict(<class 'int'>, {'The': 3, 'old': 1, 'clock': 6, 'on': 1, 'the': 5, 'wall': 1, 'ticked': 1, 'with': 1, 'a': 6, 'steady': 1, 'tick': 1, '-': 1, 'tock': 1, '.': 5, 'had': 1, 'been': 1, 'there': 1, 'for': 1, 'long': 2, ',': 4, 'time': 2, 'It': 1, 'was': 2, 'that': 2, 'kept': 1, 'perfectly': 1, 'never': 1, 'failed': 1, 'Its': 1, 'ticking': 1, 'familiar': 1, 'sound': 2, 'comforting': 1, 'in': 1, 'quiet': 1, 'room': 1, 'watched': 1, 'as': 1, 'world': 1, 'changed': 1, 'but': 1, 'itself': 1, 'remained': 1, 'same': 1})



<font face="B Nazanin" size=4>
    <div dir="rtl">
        <p>
        الفبا (تمام کاراکترهای منحصر به فرد) را از این کلمات استخراج و به عنوان واژگان اولیه (initial vocab) در نظر گرفتیم.
        </p>
    </div>

In [119]:
alphabet = []

for word in word_freqs.keys():
    for letter in word:
        if letter not in alphabet:
            alphabet.append(letter)
alphabet.sort()

In [120]:
print(alphabet)

[',', '-', '.', 'I', 'T', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'y']


In [121]:
vocab = ["<|endoftext|>"] + alphabet.copy()

In [122]:
splits = {word: [c for c in word] for word in word_freqs.keys()}


<font face="B Nazanin" size=4>
    <div dir="rtl">
        <p>
        در هر گام، فراوانی تمام زوج‌های متوالی در کلمات پیکره را محاسبه می‌کنیم.
        </p>
    </div>



In [123]:
def compute_pair_freqs(splits):
    pair_freqs = defaultdict(int)
    for word, freq in word_freqs.items():
        split = splits[word]
        if len(split) == 1:
            continue
        for i in range(len(split) - 1):
            pair = (split[i], split[i + 1])
            pair_freqs[pair] += freq
    return pair_freqs

In [124]:
pair_freqs = compute_pair_freqs(splits)

for i, key in enumerate(pair_freqs.keys()):
    print(f"{key}: {pair_freqs[key]}")
    if i >= 5:
        break

('T', 'h'): 3
('h', 'e'): 10
('o', 'l'): 1
('l', 'd'): 2
('c', 'l'): 6
('l', 'o'): 8



<font face="B Nazanin" size=4>
    <div dir="rtl">
        <p>
        زوجی که بالاترین فراوانی را دارد، به عنوان کاندیدای ادغام انتخاب می‌شود.
        </p>
    </div>



In [125]:
best_pair = ""
max_freq = None

for pair, freq in pair_freqs.items():
    if max_freq is None or max_freq < freq:
        best_pair = pair
        max_freq = freq

In [126]:
print(best_pair, max_freq)

('h', 'e') 10


In [127]:
merges = {("h", "e"): "he"}
vocab.append("he")

<font face="B Nazanin" size=4>
    <div dir="rtl">
        <p>
        merge_pair این تابع پرتکرارترین زوج را در تمام کلمات متن ادغام می‌کند.
        </p>
    </div>

In [128]:
def merge_pair(a, b, splits):
    for word in word_freqs:
        split = splits[word]
        if len(split) == 1:
            continue

        i = 0
        while i < len(split) - 1:
            if split[i] == a and split[i + 1] == b:
                split = split[:i] + [a + b] + split[i + 2 :]
            else:
                i += 1
        splits[word] = split
    return splits

In [129]:
splits = merge_pair("h", "e", splits)
print(splits["watched"])

['w', 'a', 't', 'c', 'he', 'd']


<font face="B Nazanin" size=4>
    <div dir="rtl">
        <p>
        چرخه بالا را تا رسیدن به اندازه واژگان مورد نظر (در اینجا ۵۰ توکن) ادامه می‌دهیم و تمام قوانین ادغام در دیکشنری merges ذخیره می‌شود.
        </p>
    </div>

In [130]:
vocab_size = 50

while len(vocab) < vocab_size:
    pair_freqs = compute_pair_freqs(splits)
    best_pair = ""
    max_freq = None
    for pair, freq in pair_freqs.items():
        if max_freq is None or max_freq < freq:
            best_pair = pair
            max_freq = freq
    splits = merge_pair(*best_pair, splits)
    merges[best_pair] = best_pair[0] + best_pair[1]
    vocab.append(best_pair[0] + best_pair[1])

In [131]:
print(merges)

{('h', 'e'): 'he', ('c', 'k'): 'ck', ('l', 'o'): 'lo', ('c', 'lo'): 'clo', ('clo', 'ck'): 'clock', ('t', 'he'): 'the', ('t', 'i'): 'ti', ('n', 'g'): 'ng', ('w', 'a'): 'wa', ('e', 'd'): 'ed', ('h', 'a'): 'ha', ('T', 'he'): 'The', ('ti', 'ck'): 'tick', ('o', 'r'): 'or', ('m', 'e'): 'me', ('l', 'd'): 'ld', ('i', 't'): 'it', ('r', 'e'): 're', ('f', 'or'): 'for', ('lo', 'ng'): 'long', ('ti', 'me'): 'time'}


In [132]:
print(vocab)

['<|endoftext|>', ',', '-', '.', 'I', 'T', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'y', 'he', 'ck', 'lo', 'clo', 'clock', 'the', 'ti', 'ng', 'wa', 'ed', 'ha', 'The', 'tick', 'or', 'me', 'ld', 'it', 're', 'for', 'long', 'time']


<font face="B Nazanin" size=4>
    <div dir="rtl">
        <p>
        پس از اتمام آموزش، تابع tokenize نهایی برای استفاده در متون جدید آماده شد. این تابع:


*   متن ورودی را با استفاده از همان تابع simple_pre_tokenize به توکن‌های اولیه تقسیم می‌کند.

*   تمامی قوانین ادغام ذخیره‌شده در merges را به ترتیب روی توکن‌های اولیه اعمال می‌کند تا به نمایش نهایی زیرکلمه‌ای برسد.
        </p>
    </div>

In [133]:
def tokenize(text):
    pre_tokenized_text = simple_pre_tokenize(text)
    splits = [[l for l in word] for word in pre_tokenized_text]
    for pair, merge in merges.items():
        for idx, split in enumerate(splits):
            i = 0
            while i < len(split) - 1:
                if split[i] == pair[0] and split[i + 1] == pair[1]:
                    split = split[:i] + [merge] + split[i + 2 :]
                else:
                    i += 1
            splits[idx] = split

    return sum(splits, [])

In [134]:
tokenize("The old clock on the wall ticked with a steady tick-tock.")

['The',
 'o',
 'ld',
 'clock',
 'o',
 'n',
 'the',
 'wa',
 'l',
 'l',
 'tick',
 'ed',
 'w',
 'it',
 'h',
 'a',
 's',
 't',
 'e',
 'a',
 'd',
 'y',
 'tick',
 '-',
 't',
 'o',
 'ck',
 '.']